# Role Coordination Game V2

This notebook implements a cooperative multi-agent role coordination game where three players must coordinate to defeat an enemy by adopting complementary roles (Fighter, Tank, Healer).

## Overview

**Key Features:**
- **Bayesian Role Inference**: Players use probabilistic inference to deduce each other's roles from observed actions
- **Dynamic Enemy Health**: Enemy health changes based on team composition and actions
- **Utility-Based Role Assignment**: Player stats influence role suitability
- **Multiple Variants**: Compare unique vs non-unique role constraints, basic vs enhanced combat systems

**Game Mechanics:**
- 3 Players must coordinate roles: Fighter (damage), Tank (defense), Healer (support)
- Enemy health progresses through 5 states: DEAD → LOW → MEDIUM → HIGH → FULL
- Team health progresses through 5 states: DEAD → LOW → MEDIUM → HIGH → FULL
- Team wins by reducing enemy health to DEAD
- Attacking increases probability that enemy transitions to a lower health
- Defending decreases probability that team health transitions to a lower health
- Healing increases probability that team health transitions to a higher health
- Players infer roles through Bayesian updating based on observed actions


## 1. Imports and Setup

In [12]:
from memo import memo
import jax
import jax.numpy as np
from enum import IntEnum, auto
import random

## 2. Game Definition

This section defines the core game mechanics:
- **Roles**: Fighter, Tank, Healer (each has different action preferences)
- **Actions**: Attack, Defend, Heal
- **Enemy Health States**: DEAD, LOW, MEDIUM, HIGH, FULL
- **TEAM Health States**: DEAD, LOW, MEDIUM, HIGH, FULL
- **Role Policies**: Probability distributions over actions given role and enemy health
- **Transition Function**: How enemy health changes based on number of attackers
- **Player Stats**: Each player has STR/DEF/SUP stats that influence role suitability


In [13]:
class Role(IntEnum): Fighter = 0; Tank = 1; Healer = 2
class Action(IntEnum): Attack = 0; Defend = 1; Heal = 2
class EnemyHealth(IntEnum): DEAD = 0; LOW = 1; MEDIUM = 2; HIGH = 3; FULL = 4
class TeamHealth(IntEnum): DEAD = 0; LOW = 1; MEDIUM = 2; HIGH = 3; FULL = 4  

@jax.jit
def role_policy(role, action, enemy_health, team_health):  # role-conditioned action likelihood P(action | role, enemy_health, team_health)
    return ROLE_ACTION_STATE_PROBS[role, enemy_health, team_health, action]

@jax.jit
def transition(enemy_health, team_health, actions, stats):
    # TODO: Implement transition function
    # Based on the STR/DEF/SUP stats, the team can either lose health, stay the same, or gain health
    # Based on the STR/DEF/SUP stats of each attacker, the enemy can either lose health, stay the same, or gain health
    
    return next_health



## 3. Role Inference Model

The core inference mechanism using the `memo` probabilistic programming framework. Players observe each other's actions and update their beliefs about role assignments using Bayesian inference.

**Key Function**: `role_inference()` - Updates role probability distribution given observed actions and health of the enemy and team.


In [14]:
# Due to some of memo's limitations, we also need to introduce a helper function that extracts an element from a (3D) array:
@jax.jit
def get_element(array, i0, i1, i2):
    return array[i0, i1, i2]

@memo
def role_inference[r0 : Role, r1 : Role, r2 : Role](role_prior, obs_a0, obs_a1, obs_a2, enemy_health, team_health):
    observer: knows(r0, r1, r2) # Push array axis variables into observer's frame
    observer: thinks[
        team: assigned(r0 in Role, r1 in Role, r2 in Role, wpp=get_element(role_prior, r0, r1, r2)), # Assign roles to each player
        team: chooses(a0 in Action, wpp=role_policy(r0, a0, enemy_health, team_health)), # Choose player 0's action (a0) according to their role (r0)
        team: chooses(a1 in Action, wpp=role_policy(r1, a1, enemy_health, team_health)), # Choose player 1's action (a1) according to their role (r1)
        team: chooses(a2 in Action, wpp=role_policy(r2, a2, enemy_health, team_health))  # Choose player 2's action (a2) according to their role (r2)
    ]
    observer: observes_that [team.a0 == obs_a0] # Observe player 0's action
    observer: observes_that [team.a1 == obs_a1] # Observe player 1's action
    observer: observes_that [team.a2 == obs_a2] # Observe player 2's action
    return observer[Pr[r0 == team.r0 and r1 == team.r1 and r2 == team.r2]]


## 4. Simulation Framework

Functions to simulate the coordination game over multiple timesteps:
- **`marginalize()`**: Extract single-agent role beliefs from joint distribution
- **`simulate_role_convergence()`**: Run full simulation with role inference and enemy health 

In [ ]:
def marginalize(probs, keepaxis):
    return np.sum(probs, axis=tuple(i for i in range(probs.ndim) if i != keepaxis))

def simulate_role_convergence(n_steps, player_stats, role_prior):
   # Enemy health and team health are initialized to FULL
   
   role_probs = role_prior
  # Preallocate arrays for histories
  role_history = np.zeros((n_steps, 3), dtype=np.int32)
  action_history = np.zeros((n_steps, 3), dtype=np.int32)
  role_prob_history = np.zeros((n_steps, 3, 3, 3), dtype=np.float32)
  for t in range(n_steps):
    # Sample a role for each agent according to the marginal distribution over their role
    roles = [random.choices([0, 1, 2], weights=marginalize(role_probs, i))[0] for i in range(3)]
    role_history = role_history.at[t].set(np.array(roles)) # Store roles

    # Sample an action for each agent according to the role policy
    actions = [random.choices([0, 1, 2], weights=ROLE_ACTION_PROBS[r, :])[0] for r in roles]
    action_history = action_history.at[t].set(np.array(actions)) # Store actions

    # Update (common) beliefs about role assignments via role inference
    role_probs = role_inference(role_probs, *actions)
    role_prob_history = role_prob_history.at[t].set(role_probs) # Store updated probabilities

## 5. Simulation Run